In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score
from scipy.stats.mstats import winsorize
from imblearn.over_sampling import SMOTE
import os
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

In [7]:
notebook_dir = os.getcwd()
path = os.path.join(notebook_dir, '..', 'data', 'processed', 'stroke_processed.csv')
stroke_dataset = pd.read_csv(path)

In [8]:
print(f'Shape of dataset: {stroke_dataset.shape}')
print(stroke_dataset.dtypes)
stroke_dataset.sample(10)

Shape of dataset: (4254, 12)
id                     int64
gender                object
age                  float64
hypertension            bool
heart_disease           bool
ever_married          object
work_type             object
Residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
stroke                  bool
dtype: object


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
1179,32645,Female,44.0,False,False,Yes,Private,Rural,97.59,30.5,smokes,False
3518,5223,Female,21.0,False,False,No,Private,Rural,78.32,27.0,Unknown,False
1660,8960,Female,42.0,False,False,No,Self-employed,Rural,73.41,56.0,smokes,False
3549,71297,Female,80.0,True,False,Yes,Private,Urban,125.89,28.9,smokes,False
869,67405,Female,37.0,False,False,Yes,Private,Urban,84.13,27.0,never smoked,False
3509,36298,Female,48.0,False,False,Yes,Self-employed,Rural,71.93,41.7,never smoked,False
4454,51512,Female,19.0,False,False,No,Private,Rural,57.40,22.9,Unknown,False
2991,5780,Female,47.0,False,False,Yes,Private,Urban,74.63,45.3,never smoked,False
2867,37526,Female,68.0,True,True,Yes,Private,Rural,233.30,29.2,Unknown,False
1533,31415,Female,54.0,False,False,Yes,Private,Urban,207.79,38.6,never smoked,False


In [9]:
stroke_dataset['gender'] = stroke_dataset['gender'].astype('category')
stroke_dataset['heart_disease'] = stroke_dataset['heart_disease'].astype('bool')
stroke_dataset['ever_married'] = stroke_dataset['ever_married'].astype('category')
stroke_dataset['work_type'] = stroke_dataset['work_type'].astype('category')
stroke_dataset['Residence_type'] = stroke_dataset['Residence_type'].astype('category')
stroke_dataset['smoking_status'] = stroke_dataset['smoking_status'].astype('category')
stroke_dataset['hypertension'] = stroke_dataset['hypertension'].astype('bool')
stroke_dataset['stroke'] = stroke_dataset['stroke'].astype('bool')

In [10]:
stroke_dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4254 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   id                 4254 non-null   int64   
 1   gender             4254 non-null   category
 2   age                4254 non-null   float64 
 3   hypertension       4254 non-null   bool    
 4   heart_disease      4254 non-null   bool    
 5   ever_married       4254 non-null   category
 6   work_type          4254 non-null   category
 7   Residence_type     4254 non-null   category
 8   avg_glucose_level  4254 non-null   float64 
 9   bmi                4254 non-null   float64 
 10  smoking_status     4254 non-null   category
 11  stroke             4254 non-null   bool    
dtypes: bool(3), category(5), float64(3), int64(1)
memory usage: 200.2 KB


In [14]:
print(stroke_dataset['stroke'].value_counts())
print(stroke_dataset['stroke'].value_counts(normalize=True))

stroke
False    4007
True      247
Name: count, dtype: int64
stroke
False    0.941937
True     0.058063
Name: proportion, dtype: float64


In [17]:
for column in ['hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']:
    stroke_dataset[column] = np.log1p(stroke_dataset[column])

## From chi-square test I can eliminate residence_type since it has no relationship to the target variable domain knowledge wise and test wise.